# AI Quiz Game · Gender & Faker name audit

This notebook executes the **unmodified** `generate_identity()` from [PR #11, commit 497bd20](https://github.com/Subuday77/ai-quiz-game/blob/497bd20de02745bcaf38923f47814739569527ff/backend/platinum_helper/identity.py), embedded below as retrieved from GitHub. It does not call the game's API or database.

Run **Runtime → Run all**; change the locale, sample size, or seed to repeat. Female–female in any specific two-person sequence has a 25% probability if gender is independent and fair.

In [ ]:
%pip -q install "Faker>=35,<50" pandas matplotlib

## Exact production identity generator

In [ ]:
"""Token-free, UI-locale-matched fictional identities for persisted helpers.

Only the selected *interface* language controls Faker. Question language,
profession, and the helper's stored nationality never override it.
"""
from random import Random

from faker import Faker

# Covers every language in frontend/src/app/localization/store.ts. An unknown
# language is rejected rather than silently falling back to Latin en_US.
UI_FAKER_LOCALES = {
    "en": "en_US",
    "de": "de_DE",
    "fr": "fr_FR",
    "es": "es_ES",
    "it": "it_IT",
    "pt": "pt_BR",
    "ru": "ru_RU",
    "he": "he_IL",
    "ar": "ar_SA",
    "ja": "ja_JP",
    "zh": "zh_CN",
}

PERSON_LOCALES = {
    "en_US": "United States",
    "de_DE": "Germany",
    "fr_FR": "France",
    "es_ES": "Spain",
    "it_IT": "Italy",
    "pt_BR": "Brazil",
    "ru_RU": "Russia",
    "he_IL": "Israel",
    "ar_SA": "Saudi Arabia",
    "ja_JP": "Japan",
    "zh_CN": "China",
}


def generate_identity(rng: Random | None = None, *, ui_language: str = "en") -> dict[str, str]:
    """Generate one permanent localized name; the UI never translates it."""
    if ui_language not in UI_FAKER_LOCALES:
        raise ValueError("Unsupported interface language for helper identity")
    rng = rng or Random()
    locale = UI_FAKER_LOCALES[ui_language]
    gender = rng.choice(("male", "female"))
    fake = Faker(locale)
    fake.seed_instance(rng.randrange(2**32))
    first = getattr(fake, "first_name_" + gender, fake.first_name)()
    last = getattr(fake, "last_name_" + gender, fake.last_name)()
    return {
        "name": f"{first} {last}",
        "gender": gender,
        "nationality": PERSON_LOCALES[locale],
        "locale": locale,
    }


## Experiment settings

In [ ]:
UI_LANGUAGE = "ru"  # @param ["ru","en","de","fr","es","it","pt","he","ar","ja","zh"]
SAMPLES = 10000  # @param {type:"integer"}
SEED = 20261107  # @param {type:"integer"}
assert UI_LANGUAGE in UI_FAKER_LOCALES and 100 <= SAMPLES <= 100000

## Gender distribution and actual generated names

In [ ]:
from collections import Counter
from math import erfc, sqrt
import pandas as pd
rng = Random(SEED)
people = [generate_identity(rng, ui_language=UI_LANGUAGE) for _ in range(SAMPLES)]
counts = Counter(p["gender"] for p in people)
male, female = counts["male"], counts["female"]
z = (female-SAMPLES/2)/sqrt(SAMPLES/4)
p_value = erfc(abs(z)/sqrt(2))  # Approximate two-sided p-value
print(f"Male {male:,} ({male/SAMPLES:.2%}), female {female:,} ({female/SAMPLES:.2%})")
print(f"Expected 95% sampling range: 50% ± {100*1.96*sqrt(.25/SAMPLES):.2f} percentage points")
print(f"z = {z:.2f}, approximate p-value = {p_value:.4f}")
display(pd.DataFrame(people[:20])[["name","gender","locale"]])

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7,4))
plt.bar(["male","female"],[male,female])
plt.axhline(SAMPLES/2,linestyle="--",label="Expected 50%")
plt.title(f"AI Quiz Game · {UI_LANGUAGE}, n={SAMPLES:,}")
plt.ylabel("Count")
plt.legend()
plt.show()
female_pairs = sum(people[i-1]["gender"] == people[i]["gender"] == "female" for i in range(1,len(people)))
longest = streak = 0
for p in people:
    streak = streak+1 if p["gender"]=="female" else 0
    longest = max(longest,streak)
print(f"Consecutive female pairs: {female_pairs:,}; longest female streak: {longest}")

## Russian: compare names and surnames against the *gender-specific Faker provider lists*

In [ ]:
import re
if UI_LANGUAGE == "ru":
    provider = next(p for p in Faker("ru_RU").get_providers() if p.__module__.startswith("faker.providers.person."))
    first = {g: set(getattr(provider,"first_names_"+g)) for g in ("male","female")}
    last = {g: set(getattr(provider,"last_names_"+g)) for g in ("male","female")}
    problems = []
    for person in people:
        full, g = person["name"], person["gender"]
        name, surname = full.split(" ",1)
        if not re.fullmatch(r"[А-Яа-яЁё -]+",full) or name not in first[g] or surname not in last[g]:
            problems.append(person)
    print(f"Wrong script or gender-inconsistent according to Faker: {len(problems)} / {len(people)}")
    if problems: display(pd.DataFrame(problems[:15]))
else:
    print("Set UI_LANGUAGE='ru' for Russian-only validation.")

## Quick check of all 11 interface locales

In [ ]:
scripts = {"ru":r"[А-Яа-яЁё]","he":r"[\u0590-\u05ff]","ar":r"[\u0600-\u06ff]","ja":r"[\u3040-\u30ff\u4e00-\u9fff]","zh":r"[\u4e00-\u9fff]"}
rows=[]
for language, locale in UI_FAKER_LOCALES.items():
    r=Random(SEED)
    sample=[generate_identity(r,ui_language=language) for _ in range(200)]
    off_script=sum(not re.search(scripts[language],p["name"]) for p in sample) if language in scripts else None
    rows.append({"UI":language,"Faker locale":locale,"male":sum(p["gender"]=="male" for p in sample),"female":sum(p["gender"]=="female" for p in sample),"off script":off_script,"example":sample[0]["name"]})
display(pd.DataFrame(rows))

**Interpretation:** Two female helpers in a row are not evidence of bias; a large repeatable deviation from 50/50 would be. The Russian list checks the provider's definition of male/female names, not assumptions about Russian suffixes. Existing purchased helpers are persisted and will not be rerolled. This notebook changes no game code.